# Fase 1 — Sección 5: Correcciones
**Proyecto Final — Gestión de Datos (UAX 2025/2026)**

Aplicación de las 3 correcciones de severidad Alta identificadas en el inventario.
Cada corrección incluye: justificación, snapshot pre-corrección, UPDATE y verificación inmediata.

| ID | Tabla | Corrección | Orden |
|----|-------|-----------|-------|
| P06 | `product` | Precio producto 29: 19.99€ → 99.90€ | 1º |
| P07 | `sale_item` | Subtotales inconsistentes (8 líneas) | 2º |
| P08 | `sale` | Total venta 13009 recalculado | 3º |

> ⚠️ **Regla de negocio:** nunca eliminar clientes. Ninguna de estas correcciones afecta a registros de cliente.

---

## Configuración

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from pathlib import Path
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

PROJECT_ROOT = Path(r'C:\Users\Propietario\Desktop\segundo cuatri\gestion de datos\trabajo final gd')

print('Librerias importadas.')

Librerias importadas.


In [2]:
DB_USER     = 'postgres'
DB_PASSWORD = 'pimpum.postgre'
DB_HOST     = 'localhost'
DB_PORT     = '5432'
DB_NAME     = 'saleshealth'

DATABASE_URL = f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).fetchone()[0]

print(f'Conexion establecida. PostgreSQL: {version.split(",")[0]}')

def query(sql, params=None):
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn, params=params)

def execute(sql, params=None):
    with engine.begin() as conn:
        result = conn.execute(text(sql), params or {})
        return result.rowcount

Conexion establecida. PostgreSQL: PostgreSQL 18.3 on x86_64-windows


---
## 5.0 — Snapshot pre-corrección
Guardamos el estado actual de todos los registros que vamos a modificar.
Si algo falla, este snapshot permite revertir los cambios manualmente.

In [3]:
# ── Snapshot producto 29 ──────────────────────────────────────────────────────
snap_p06 = query("SELECT product_id, name, price FROM product WHERE product_id = 29")
print('SNAPSHOT — product (producto 29):')
display(snap_p06)

# ── Snapshot sale_item inconsistentes ────────────────────────────────────────
snap_p07 = query("""
    SELECT sale_item_id, sale_id, product_id, quantity, unit_price, subtotal,
           ROUND(quantity * unit_price, 2) AS subtotal_correcto
    FROM sale_item
    WHERE ABS(subtotal - ROUND(quantity * unit_price, 2)) > 0.01
    ORDER BY sale_item_id
""")
print('\nSNAPSHOT — sale_item (lineas con subtotal incorrecto):')
display(snap_p07)

# ── Snapshot venta 13009 ──────────────────────────────────────────────────────
snap_p08 = query("""
    SELECT s.sale_id, s.total AS total_actual,
           SUM(si.subtotal) AS suma_subtotales
    FROM sale s
    JOIN sale_item si ON s.sale_id = si.sale_id
    WHERE s.sale_id = 13009
    GROUP BY s.sale_id, s.total
""")
print('\nSNAPSHOT — sale (venta 13009):')
display(snap_p08)

print('\n✅ Snapshot pre-correccion guardado correctamente.')
print(f'   Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

SNAPSHOT — product (producto 29):


,product_id,name,price
0,29,Sensor temperatura inteligente,19.99



SNAPSHOT — sale_item (lineas con subtotal incorrecto):


,sale_item_id,sale_id,product_id,quantity,unit_price,subtotal,subtotal_correcto
0,10332,4137,5,1,79.99,71.99,79.99
1,10987,4403,3,2,24.90,47.31,49.80
2,21846,8715,1,2,59.99,113.98,119.98
3,22613,9037,1,2,59.99,113.98,119.98
4,24587,9831,3,3,24.90,72.21,74.70
5,26073,10429,15,2,89.90,170.81,179.80
6,32543,13009,2,1,29.99,26.99,29.99
7,32544,13009,30,2,99.90,189.81,199.80



SNAPSHOT — sale (venta 13009):


,sale_id,total_actual,suma_subtotales
0,13009,321.77,321.77



✅ Snapshot pre-correccion guardado correctamente.
   Timestamp: 2026-05-02 14:52:32


---
## 5.1 — Corrección P06: Precio producto 29

**Problema:** El producto 29 "Sensor temperatura inteligente" tiene precio de venta 19.99€ con coste 59.94€, generando un margen de -199.85%.

**Justificación:** Todos los productos de la categoría 1 (Diagnóstico) tienen margen fijo del 40%.
Precio correcto = `unit_cost / (1 - 0.40) = 59.94 / 0.60 = 99.90€`

**Impacto:** 1 registro en `product`

In [4]:
# ── Aplicar corrección P06 ────────────────────────────────────────────────────
PRECIO_CORRECTO = 99.90

filas = execute(
    "UPDATE product SET price = :precio WHERE product_id = 29",
    {'precio': PRECIO_CORRECTO}
)

print('=' * 52)
print('  CORRECCION P06 — product.price')
print('=' * 52)
print(f'  Registros actualizados : {filas}')
print(f'  Valor anterior         : 19.99 EUR')
print(f'  Valor nuevo            : {PRECIO_CORRECTO} EUR')
print('=' * 52)

# ── Verificacion inmediata ────────────────────────────────────────────────────
verificacion = query("""
    SELECT p.product_id, p.name, p.price, cp.unit_cost,
           ROUND((p.price - cp.unit_cost) / p.price * 100, 2) AS margen_pct
    FROM product p
    JOIN central_product cp ON p.product_id = cp.product_id
    WHERE p.product_id = 29
""")

margen = float(verificacion['margen_pct'].iloc[0])
precio = float(verificacion['price'].iloc[0])

print(f'\n  Verificacion post-correccion:')
print(f'  Precio actualizado : {precio:.2f} EUR')
print(f'  Margen resultante  : {margen:.2f}%', end='')
print('  ✅ OK' if abs(margen - 40.0) < 0.5 else '  ⚠  Revisar')

  CORRECCION P06 — product.price
  Registros actualizados : 1
  Valor anterior         : 19.99 EUR
  Valor nuevo            : 99.9 EUR

  Verificacion post-correccion:
  Precio actualizado : 99.90 EUR
  Margen resultante  : 40.00%  ✅ OK


---
## 5.2 — Corrección P07: Subtotales inconsistentes en sale_item

**Problema:** 8 líneas de venta tienen `subtotal` que no coincide con `quantity × unit_price`. Diferencia total: -46.96 EUR.

**Justificación:** `subtotal` es un campo derivado. Los ítems (`quantity` y `unit_price`) son la fuente de verdad.

**Impacto:** 8 registros en `sale_item`

In [5]:
# ── Aplicar corrección P07 ────────────────────────────────────────────────────
filas = execute("""
    UPDATE sale_item
    SET subtotal = ROUND(quantity * unit_price, 2)
    WHERE ABS(subtotal - ROUND(quantity * unit_price, 2)) > 0.01
""")

print('=' * 52)
print('  CORRECCION P07 — sale_item.subtotal')
print('=' * 52)
print(f'  Registros actualizados : {filas}')
print(f'  Criterio               : subtotal = ROUND(quantity * unit_price, 2)')
print('=' * 52)

# ── Verificacion inmediata ────────────────────────────────────────────────────
restantes = query("""
    SELECT COUNT(*) AS n
    FROM sale_item
    WHERE ABS(subtotal - ROUND(quantity * unit_price, 2)) > 0.01
""")['n'].iloc[0]

print(f'\n  Verificacion post-correccion:')
print(f'  Lineas aun inconsistentes : {int(restantes)}', end='')
print('  ✅ OK' if restantes == 0 else '  ⚠  Revisar')

  CORRECCION P07 — sale_item.subtotal
  Registros actualizados : 8
  Criterio               : subtotal = ROUND(quantity * unit_price, 2)

  Verificacion post-correccion:
  Lineas aun inconsistentes : 0  ✅ OK


---
## 5.3 — Corrección P08: Total venta 13009

**Problema:** El campo `total` de la venta 13009 no coincide con la suma de sus `sale_item.subtotal`.

**Justificación:** `total` es un campo derivado. Se recalcula desde los subtotales ya corregidos en P07.

**Nota:** Esta corrección se ejecuta **después de P07** para usar los subtotales ya corregidos.

**Impacto:** 1 registro en `sale`

In [8]:
# ── Aplicar corrección P08 — TODOS los totales inconsistentes ─────────────────
# Primero vemos cuantas ventas tienen total incorrecto
df_inconsistentes = query("""
    SELECT s.sale_id, s.total,
           ROUND(SUM(si.subtotal), 2)           AS suma_subtotales,
           s.total - ROUND(SUM(si.subtotal), 2) AS diferencia
    FROM sale s
    JOIN sale_item si ON s.sale_id = si.sale_id
    GROUP BY s.sale_id, s.total
    HAVING ABS(s.total - SUM(si.subtotal)) > 0.01
    ORDER BY s.sale_id
""")

print('Ventas con total inconsistente antes de la correccion:')
display(df_inconsistentes)

# Corregimos todas de una vez
filas = execute("""
    UPDATE sale s
    SET total = sub.suma
    FROM (
        SELECT sale_id, ROUND(SUM(subtotal), 2) AS suma
        FROM sale_item
        GROUP BY sale_id
    ) sub
    WHERE s.sale_id = sub.sale_id
      AND ABS(s.total - sub.suma) > 0.01
""")

print('=' * 52)
print('  CORRECCION P08 — sale.total')
print('=' * 52)
print(f'  Registros actualizados : {filas}')
print(f'  Criterio               : total = ROUND(SUM(sale_item.subtotal), 2)')
print('=' * 52)

# Verificacion inmediata
restantes = query("""
    SELECT COUNT(*) AS n
    FROM sale s
    JOIN sale_item si ON s.sale_id = si.sale_id
    GROUP BY s.sale_id, s.total
    HAVING ABS(s.total - SUM(si.subtotal)) > 0.01
""")

print(f'\n  Verificacion post-correccion:')
print(f'  Ventas aun inconsistentes : {len(restantes)}', end='')
print('  ✅ OK' if len(restantes) == 0 else '  ⚠  Revisar')

Ventas con total inconsistente antes de la correccion:


,sale_id,total,suma_subtotales,diferencia
0,4137,441.67,449.67,-8.00
1,4403,47.31,49.80,-2.49
2,8715,363.57,369.57,-6.00
3,9037,893.91,899.91,-6.00
4,9831,72.21,74.70,-2.49
5,10429,530.76,539.75,-8.99


  CORRECCION P08 — sale.total
  Registros actualizados : 6
  Criterio               : total = ROUND(SUM(sale_item.subtotal), 2)

  Verificacion post-correccion:
  Ventas aun inconsistentes : 0  ✅ OK


---
## 5.4 — Registro de cambios aplicados

In [10]:
# ── Log de correcciones ───────────────────────────────────────────────────────
log = [
    {
        'id'           : 'P06',
        'tabla'        : 'product',
        'columna'      : 'price',
        'registros'    : 1,
        'valor_antes'  : '19.99 EUR',
        'valor_despues': '99.90 EUR',
        'justificacion': 'Margen fijo 40% en categoria 1 → price = 59.94 / 0.60',
        'estado'       : 'Aplicado'
    },
    {
        'id'           : 'P07',
        'tabla'        : 'sale_item',
        'columna'      : 'subtotal',
        'registros'    : 8,
        'valor_antes'  : 'subtotal incorrecto (-46.96 EUR total)',
        'valor_despues': 'subtotal = ROUND(quantity * unit_price, 2)',
        'justificacion': 'subtotal es campo derivado — items son fuente de verdad',
        'estado'       : 'Aplicado'
    },
    {
        'id'           : 'P08',
        'tabla'        : 'sale',
        'columna'      : 'total',
        'registros'    : 6,
        'valor_antes'  : 'total != SUM(subtotales) en 6 ventas',
        'valor_despues': 'total = ROUND(SUM(sale_item.subtotal), 2)',
        'justificacion': 'total es campo derivado — ejecutado despues de P07',
        'estado'       : 'Aplicado'
    },
]

df_log = pd.DataFrame(log)

print('=' * 72)
print(f'  LOG DE CORRECCIONES — {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 72)
for _, row in df_log.iterrows():
    print(f"\n  [{row['id']}] {row['tabla']}.{row['columna']}  —  {row['registros']} registro(s)")
    print(f"  Antes        : {row['valor_antes']}")
    print(f"  Despues      : {row['valor_despues']}")
    print(f"  Justificacion: {row['justificacion']}")
    print(f"  Estado       : ✅ {row['estado']}")
print(f"\n{'='*72}")
print(f"  Total correcciones aplicadas: {len(df_log)}")
print(f"  Registros modificados       : {df_log['registros'].sum()}")
print('=' * 72)
print('\nSeccion 5 completada — correcciones aplicadas correctamente.')

  LOG DE CORRECCIONES — 2026-05-02 15:09:49

  [P06] product.price  —  1 registro(s)
  Antes        : 19.99 EUR
  Despues      : 99.90 EUR
  Justificacion: Margen fijo 40% en categoria 1 → price = 59.94 / 0.60
  Estado       : ✅ Aplicado

  [P07] sale_item.subtotal  —  8 registro(s)
  Antes        : subtotal incorrecto (-46.96 EUR total)
  Despues      : subtotal = ROUND(quantity * unit_price, 2)
  Justificacion: subtotal es campo derivado — items son fuente de verdad
  Estado       : ✅ Aplicado

  [P08] sale.total  —  6 registro(s)
  Antes        : total != SUM(subtotales) en 6 ventas
  Despues      : total = ROUND(SUM(sale_item.subtotal), 2)
  Justificacion: total es campo derivado — ejecutado despues de P07
  Estado       : ✅ Aplicado

  Total correcciones aplicadas: 3
  Registros modificados       : 15

Seccion 5 completada — correcciones aplicadas correctamente.
